# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring

**Primary ML task type: Ranking**

The goal is to rank content pages by how strongly they should be prioritized for human review. This is more naturally a ranking problem than a simple classification problem because the final decision is not only whether a page is an opportunity, but **which pages should be reviewed first when review capacity is limited**.

A model would produce a score for each page, and pages could then be ordered from highest to lowest priority. The highest-ranked pages would form a review queue for potential refresh, expansion, protection, pruning, or monitoring.

In [2]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Unit of analysis: one content page")
print("Unique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)
Unit of analysis: one content page
Unique content pages: 30000
Unique clients: 32


## 2. Target or proxy

The eventual target should be a **future-looking page outcome**, rather than the starter dataset's current-window `trend_direction`.

A useful proxy target would be:

**`future_decline` = 1 if the page experiences a meaningful deterioration in search visibility during a future outcome window, otherwise 0.**

The exact threshold and time window should be defined during the later data-contract stage after checking the available time fields and avoiding leakage.

For the current starter data, `trend_direction` can be used as an exploratory proxy because it provides an observed distinction between pages moving up, down, flat, new, or stable. However, I will not treat it as the final predictive target.

The eventual ranking score would estimate how strongly each page should be prioritized for review based on the probability or expected value of the defined future outcome.

In [5]:
# Current starter proxy: observed trend direction.
# This is for task exploration only, not the final future target.

proxy_counts = (
    df["trend_direction"]
      .value_counts(dropna=False)
      .rename_axis("trend_direction")
      .to_frame("count")
)

proxy_counts["share"] = (
    proxy_counts["count"] / len(df)
).round(3)

display(proxy_counts)
print(
    "\nNote: trend_direction is an observed current-window proxy, "
    "not the final future-looking prediction target."
)

,count,share
trend_direction,,
down,16262,0.542
stable,5962,0.199
up,4388,0.146
new,2236,0.075
flat,1152,0.038



Note: trend_direction is an observed current-window proxy, not the final future-looking prediction target.


## 3. Success metric

### Primary metric: Precision@K

The primary success metric will be **Precision@K**, where K represents the number of pages the team can realistically review.

For example, Precision@20 measures the proportion of the top 20 ranked pages that meet the defined positive outcome criteria.

This metric fits the business decision because review capacity is limited. If a team can only investigate a small number of pages, the quality of the highest-ranked pages matters more than performance across every page.

I would also examine Precision@50 and compare the model against a simple baseline or fixed rule.

### What good looks like

A useful model should produce a review queue whose top-ranked pages contain a higher proportion of relevant opportunities than a simple fixed rule or naive baseline.

The metric should be evaluated on held-out data and, later, under client-aware validation so that pages from the same client do not create an overly optimistic estimate.

In [6]:
# Example of how Precision@K works for a review queue.

def precision_at_k(y_true, ranked_scores, k):
    ranked_idx = pd.Series(ranked_scores).sort_values(
        ascending=False
    ).index[:k]

    return pd.Series(y_true).iloc[ranked_idx].mean()


# Example:
example_actual = [1, 0, 1, 1, 0, 0, 1, 0, 0, 1]
example_score = [0.95, 0.90, 0.88, 0.80, 0.70, 0.60, 0.55, 0.40, 0.30, 0.10]

print(
    "Example Precision@5:",
    precision_at_k(example_actual, example_score, 5)
)

Example Precision@5: 0.6


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content page**.

Each row represents one content item for one client, with page-level attributes and performance signals such as content age, days since update, impressions, clicks, sessions, and average search position.

The eventual model will assign one ranking score to each page. Those scores can then be aggregated into a client-level review queue, but the prediction unit itself remains the individual content page.

In [7]:
# Show the actual page-level unit of analysis.

page_sample = df[
    [
        "client_id",
        "content_id",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "avg_position",
        "trend_direction",
    ]
].head(10)

display(page_sample)

print("Rows shown:", len(page_sample))
print("Unique content IDs:", page_sample["content_id"].nunique())

,client_id,content_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,avg_position,trend_direction
0,client_f369cb89fc,content_304f48230142,187,20,3803,29,17,10.6,down
1,client_4e07408562,content_a1fb4e703a9e,445,25,15320,7,9,20.3,down
2,client_7f2253d7e2,content_9aa793d4d895,141,20,12581,11,11,36.5,down
3,client_19581e27de,content_331d6c4de07b,463,22,11751,58,78,6.2,stable
4,client_3fdba35f04,content_d99b7a2d90ca,263,14,19140,24,145,44.0,down
5,client_f369cb89fc,content_d4084a4bc775,147,20,3970,1,5,8.5,down
6,client_8722616204,content_9a34b442b552,90,20,20,0,1,7.0,down
7,client_19581e27de,content_a63219c6e95a,445,22,1724,1,28,21.2,stable
8,client_6208ef0f77,content_5e6c160719bc,90,20,32574,29,68,46.0,down
9,client_19581e27de,content_c27558df2b0c,257,104,1240,2,3,4.9,down


Rows shown: 10
Unique content IDs: 10


## 5. Why ML beats a fixed rule here

A fixed rule can be useful as a baseline, but it requires manually choosing thresholds and usually treats signals independently. For example, a rule might prioritize every page older than a certain number of days.

The opportunity for ML is to combine multiple signals and learn how their combinations relate to the defined outcome. A page with older content may not deserve the same priority if it has very different impressions, search position, traffic, or recent-update behavior.

A ranking model can produce a continuous priority score rather than a single yes/no decision. This allows the review team to focus on the highest-ranked pages and adjust the review capacity through K.

The model will still need to beat a simple fixed rule on held-out data to justify its additional complexity. If it does not, the simpler rule may be preferable.

In [8]:
# Observed starter experiment:
# Compare the simple hand rule with the readable decision tree.

results = pd.DataFrame({
    "metric": ["Precision@20", "Precision@50"],
    "hand_rule": [0.500, 0.540],
    "decision_tree": [0.750, 0.680]
})

results["tree_minus_rule"] = (
    results["decision_tree"] - results["hand_rule"]
).round(3)

display(results)

print(
    "In the starter held-out experiment, the readable tree "
    "outperformed the hand rule at both tested K values."
)

,metric,hand_rule,decision_tree,tree_minus_rule
0,Precision@20,0.50,0.75,0.25
1,Precision@50,0.54,0.68,0.14


In the starter held-out experiment, the readable tree outperformed the hand rule at both tested K values.


## 6. Self-check

* [x] I identified the task as a ranking problem.
* [x] I defined the prediction unit as one content page.
* [x] I identified a future-looking target concept rather than treating the starter label as the final target.
* [x] I defined Precision@K as the primary success metric.
* [x] I connected the metric to limited review capacity.
* [x] I explained what action the ranking supports.
* [x] I explained why a fixed rule is a useful baseline but may be limited.
* [x] I showed the unit of analysis using an actual dataframe.
* [x] I used the starter experiment to provide measured evidence rather than claiming guaranteed performance.
* [x] I will avoid using `trend_direction` or `trend_pct` as model features.
* [x] I will revisit the target definition and leakage controls during the data-contract stage.